# Text Emotions Classification
Text emotions classification is a natural language processing and text classification problem. 
Here, I will train a text classification model to classify the emotion of a text correctly.

In [1]:
pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 25.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 83.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 1.4 MB/s eta 0:00:000:00:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,Flatten,Dense
from IPython.display import  display
import joblib
import gradio as gr

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/emotions-dataset-for-nlp/val.txt
/kaggle/input/emotions-dataset-for-nlp/test.txt
/kaggle/input/emotions-dataset-for-nlp/train.txt


In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("praveengovi/emotions-dataset-for-nlp")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/emotions-dataset-for-nlp


In [6]:
data = pd.read_csv("/kaggle/input/emotions-dataset-for-nlp/train.txt", sep = ';')

In [7]:
data.columns = ["Text","Emotions"]
display(data.head())

,Text,Emotions
0,i can go from feeling so hopeless to so damned...,sadness
1,im grabbing a minute to post i feel greedy wrong,anger
2,i am ever feeling nostalgic about the fireplac...,love
3,i am feeling grouchy,anger
4,ive been feeling a little burdened lately wasn...,sadness


# Tokenize the data

In [8]:
texts = data['Text'].tolist()
labels = data['Emotions'].tolist()

In [9]:
#Tokenize the text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

Now we need to pad the sequences to the same length to feed them into a neural network. Here’s how we can pad the sequences of the texts to have the same length:

In [10]:
sequences = tokenizer.texts_to_sequences(texts)
max_length = max([len(seq) for seq in sequences])
padded_sequence = pad_sequences(sequences,maxlen = max_length )

Now I’ll use the label encoder method to convert the classes from strings to a numerical representation:

In [11]:
#Encode the string labels to integers
label_encoders = LabelEncoder()
labels = label_encoders.fit_transform(labels)

We are now going to One-hot encode the labels. One hot encoding refers to the transformation of categorical labels into a binary representation where each label is represented as a vector of all zeros except a single 1. This is necessary because machine learning algorithms work with numerical data. So here is how we can One-hot encode the labels:

In [12]:
#One-hot encoding the labels
one_hot_labels = tensorflow.keras.utils.to_categorical(labels)

# Text Emotions Classification Model
Now we will split the data into training and test sets:

In [13]:
X_train, X_test, y_train, y_test= train_test_split(padded_sequence,one_hot_labels, test_size = 0.2)

Now let’s define a neural network architecture for our classification problem and use it to train a model to classify emotions:

In [14]:
model = Sequential()
model.add(Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=128))
model.add(Flatten())
model.add(Dense(units= 128, activation = "relu"))
model.add(Dense(units=len(one_hot_labels[0]), activation ="softmax"))
model.compile(optimizer = 'adam',loss = 'categorical_crossentropy',metrics = ['accuracy'])
model.fit(X_train, y_train, epochs = 10,batch_size = 32, validation_data = (X_test, y_test))

Epoch 1/10


2025-05-02 00:03:12.157242: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


400/400 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.3821 - loss: 1.5246 - val_accuracy: 0.7031 - val_loss: 0.8615
Epoch 2/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.8599 - loss: 0.4423 - val_accuracy: 0.8116 - val_loss: 0.5730
Epoch 3/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.9849 - loss: 0.0662 - val_accuracy: 0.8234 - val_loss: 0.5612
Epoch 4/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9956 - loss: 0.0218 - val_accuracy: 0.8178 - val_loss: 0.6285
Epoch 5/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9970 - loss: 0.0151 - val_accuracy: 0.8188 - val_loss: 0.6089
Epoch 6/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.9967 - loss: 0.0140 - val_accuracy: 0.8181 - val_loss: 0.6553
Epoch 7/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.9976 - loss: 0.0113 - val_accuracy: 0.8138 - val_loss: 0.7300
Epoch 8/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9979 - loss: 0.0109 - val_accuracy: 0.81

In [15]:
joblib.dump(model,"emotion_model.pkl")

['emotion_model.pkl']

In [16]:
def classify_text(input_text):
    # Preprocess
    input_sequence = tokenizer.texts_to_sequences([input_text])
    padded_input_sequence = pad_sequences(input_sequence, maxlen=max_length)
    
    # Predict
    model = joblib.load('emotion_model.pkl')
    prediction = model.predict(padded_input_sequence)
    predicted_label = label_encoders.inverse_transform([np.argmax(prediction[0])])
    return predicted_label[0]

# Gradio interface
app = gr.Interface(
    fn=classify_text,
    inputs=gr.Textbox(lines=2, placeholder="Enter text here..."),
    outputs="text",
    title="Text Classification Model",
    description="Enter a sentence and get the predicted label."
)

app.launch()


* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://e1b190a2d3161b1136.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 407ms/step
